# Transformer Decoder

## Code

In [ ]:
import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn
import models.deep_learning.components as comp

## Testing

In [ ]:
# input parameters
N = 3
M = 4
batch_size = 2
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float32

# Decoder Layer parameters
d_model = 4
nhead = 2
dim_feedforward = 64
dropout = 0.2
layer_norm_eps = 1e-5
batch_first = True
norm_first = True
bias = True

tgt_mask = comp.create_causal_mask((N, N), device=device)
memory_mask = comp.create_random_mask((N, M), device=device)


## Transformer decoder parameters
num_layers = 5
#  It helps only when norm_first is True,
norm = None  # nn.LayerNorm(d_model).to(device=device,dtype=dtype)

In [ ]:
torch.manual_seed(0)
x = torch.randn(batch_size, N, d_model, device=device, dtype=dtype)
memory = torch.randn(batch_size, M, d_model, device=device, dtype=dtype)

In [ ]:
init_seed = 42  # avoide weights initialization randomness effects
train_seed = 24  # avoid dropout randomness effects

In [ ]:
torch.manual_seed(init_seed)
tf_decl = mynn.TransformerDecoderLayer(
    d_model,
    nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation_cls=nn.GELU,
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)

torch.manual_seed(init_seed)
nn_tf_decl = nn.TransformerDecoderLayer(
    d_model,
    nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation="gelu",
    layer_norm_eps=layer_norm_eps,
    batch_first=batch_first,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)

tf_dec = mynn.TransformerDecoder(tf_decl, num_layers, norm=norm)
nn_tf_dec = nn.TransformerDecoder(
    nn_tf_decl,
    num_layers,
    norm=norm,
)

tf_dec.load_weights_from_torch_decoder(nn_tf_dec)

### Evaluation

In [ ]:
tf_dec.eval()
tf_dec(x, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)

In [ ]:
nn_tf_dec.eval()
nn_tf_dec(x, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)

### Training

In [ ]:
mse = torch.nn.MSELoss()

In [ ]:
torch.manual_seed(train_seed)
nn_tf_dec.train()
out = nn_tf_dec(x, memory)
print(out)
loss = mse(out, x)
loss.backward()
optimizer = torch.optim.SGD(nn_tf_dec.parameters(), lr=1e-3)
optimizer.step()
nn_tf_dec(x, memory)


In [ ]:
torch.manual_seed(train_seed)
tf_dec.train()
out = tf_dec(x, memory)
print(out)
loss = mse(out, x)
loss.backward()
optimizer = torch.optim.SGD(tf_dec.parameters(), lr=1e-3)
optimizer.step()
tf_dec(x, memory)